<a href="https://colab.research.google.com/github/mehkzhra/FlyRank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

**Lane 2: Refresh / Content Opportunity Scoring**

This notebook maps my chosen lane onto the ML loop. The purpose is decision support for content editors, not automatic publishing or a claim that a refresh causes better performance.

## 1. My lane as an ML task (type)

My primary task is **scoring that produces a ranking**. For each measurable content page, the system would estimate a review-priority score. Sorting the scores from highest to lowest creates a queue answering: **which pages should an editor review first?**

The score would use a classification-style estimate of decline risk together with opportunity signals such as exposure. The output is a ranked queue rather than an automatic content decision. A content editor or SEO analyst reviews the top pages, examines reason codes, and chooses whether to refresh, expand, protect, prune, or monitor each page. A wrong high-priority call wastes editor time; a missed valuable declining page may leave a larger opportunity unattended.

In [1]:
task_frame = {
    "lane": "Refresh / Content Opportunity Scoring",
    "task_type": "Scoring that produces a ranking",
    "decision": "Which measurable content pages should be reviewed first?",
    "actor": "Content editor or SEO analyst",
    "output": "Ranked review queue with reason codes",
    "supported_actions": ["refresh", "expand", "protect", "prune", "monitor"],
}

for key, value in task_frame.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

Lane: Refresh / Content Opportunity Scoring
Task Type: Scoring that produces a ranking
Decision: Which measurable content pages should be reviewed first?
Actor: Content editor or SEO analyst
Output: Ranked review queue with reason codes
Supported Actions: ['refresh', 'expand', 'protect', 'prune', 'monitor']


## 2. Target or proxy

The starter data does not contain the result of an editor's intervention, so it cannot directly label "a refresh will succeed." My provisional target is therefore an **observed decline proxy**: `is_declining_proxy = 1` when impressions in the most recent 30 days are more than 20% below the previous 30 days, and `0` otherwise. The underlying measurements are observed, but the 20% cutoff is an operational definition supplied by the dataset.

This proxy is suitable for learning how to surface pages currently showing meaningful decline, but it is not causal evidence and not a forward-looking outcome. In later warehouse work, the stronger target would use an earlier feature window and an observed decline in a later window. I will never use `trend_direction` or `trend_pct` as model features because they define the proxy and would leak the answer.

In [2]:
import pandas as pd

# Sketch of the target column before loading the full lane slice.
target_sketch = pd.DataFrame({
    "impressions_prev_30d": [1000, 1000, 1000, 0],
    "impressions_last_30d": [650, 900, 1250, 100],
    "observed_change": ["-35%", "-10%", "+25%", "new page"],
    "is_declining_proxy": [1, 0, 0, 0],
})
target_sketch

,impressions_prev_30d,impressions_last_30d,observed_change,is_declining_proxy
0,1000,650,-35%,1
1,1000,900,-10%,0
2,1000,1250,+25%,0
3,0,100,new page,0


## 3. Success metric

My primary metric is **Precision@50**: among the 50 pages placed at the top of the review queue, what proportion has the observed decline proxy? This matches the real constraint that an editor can review only a limited number of pages.

For this framing stage, I define "good" as **Precision@50 ≥ 0.70**, meaning at least 35 of the first 50 pages are observed decline cases. I will also compare that result with the decline base rate in the eligible slice; 0.70 is useful only if it is meaningfully above that base rate and holds on client-grouped validation data. Precision alone does not prove that refreshing those pages will improve results.

In [3]:
def precision_at_k(y_true, priority_score, k=50):
    """Share of observed positive cases among the k highest scores."""
    evaluation = pd.DataFrame({"target": y_true, "score": priority_score})
    evaluation = evaluation.sort_values("score", ascending=False)
    return evaluation.head(k)["target"].mean()

success_metric = "Precision@50"
success_threshold = 0.70
required_declining_pages = int(50 * success_threshold)
print(f"Primary metric: {success_metric}")
print(f"Provisional success threshold: {success_threshold:.0%}")
print(f"Equivalent result: at least {required_declining_pages} observed decline cases in the top 50")

Primary metric: Precision@50
Provisional success threshold: 70%
Equivalent result: at least 35 observed decline cases in the top 50


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one anonymized content page**. I load the starter CSV and keep the lane's measurable-opportunity slice: pages with at least 100 trailing-90-day impressions and more than 0 sessions. Each displayed row is one page that could enter the editor's review queue. IDs are retained only for identification, grouping, and later client-holdout validation; they are not model features.

In [4]:
from pathlib import Path

possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
data_path = next((path for path in possible_paths if path.exists()), None)

if data_path is not None:
    raw = pd.read_csv(data_path)
else:
    data_url = (
        "https://raw.githubusercontent.com/mehkzhra/"
        "FlyRank-ML-Internship/main/data/raw/content_refresh_anonymized.csv"
    )
    raw = pd.read_csv(data_url)

lane_df = raw.loc[
    raw["impressions_90d"].ge(100) & raw["sessions_90d"].gt(0)
].copy()
lane_df["is_declining_proxy"] = lane_df["trend_direction"].eq("down").astype(int)

unit_columns = [
    "content_id", "client_id", "content_type",
    "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "sessions_90d",
    "ctr", "avg_position", "engagement_rate",
    "is_declining_proxy",
]
unit_df = lane_df[unit_columns]

assert raw.shape == (30000, 44)
assert unit_df["content_id"].is_unique
assert len(unit_df) == 22006

print(f"Starter data: {len(raw):,} rows × {raw.shape[1]} columns")
print(f"Eligible lane slice: {len(unit_df):,} content pages")
print(f"Observed decline base rate in slice: {unit_df['is_declining_proxy'].mean():.2%}")
print("Unit of analysis: one row = one anonymized content page")
unit_df.head(8)

Starter data: 30,000 rows × 44 columns
Eligible lane slice: 22,006 content pages
Observed decline base rate in slice: 59.77%
Unit of analysis: one row = one anonymized content page


,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,engagement_rate,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3803,29,17,0.76,10.6,5.88,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,15320,7,9,0.05,20.3,0.00,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,12581,11,11,0.09,36.5,0.00,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,11751,58,78,0.49,6.2,1.28,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,19140,24,145,0.13,44.0,0.00,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,3970,1,5,0.03,8.5,0.00,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,1724,1,28,0.06,21.2,3.57,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,20,32574,29,68,0.09,46.0,5.88,1


## 5. Why ML beats a fixed rule here

A fixed rule is an important baseline, but a single if-statement is unlikely to prioritize this queue well. Pages differ simultaneously in exposure, search position, click-through rate, engagement, age, update recency, content type, and missing-data patterns. The same change can matter differently for a high-impression page near page one than for a low-impression page in a deep position. These signals can interact nonlinearly, and useful thresholds may differ across clients and content types.

ML earns a place only if it learns these measured interactions and improves Precision@50 over a transparent rule baseline on held-out clients. The result remains directional decision support: the editor reviews the evidence and chooses the action. If ML does not provide a stable, explainable improvement, the simpler fixed rule should be kept.

In [5]:
# Show that observed decline cases span different page conditions.
# This is a framing check, not evidence that these fields cause decline.
signal_check = (
    lane_df.assign(
        position_band=pd.cut(
            lane_df["avg_position"].replace(0, pd.NA),
            bins=[0, 10, 20, 50, float("inf")],
            labels=["page_1", "striking", "page_3_5", "deep"],
        )
    )
    .groupby("position_band", observed=True)
    .agg(
        pages=("content_id", "size"),
        median_impressions=("impressions_90d", "median"),
        median_days_since_update=("days_since_last_update", "median"),
        observed_decline_rate=("is_declining_proxy", "mean"),
    )
    .reset_index()
)
signal_check["observed_decline_rate"] = signal_check["observed_decline_rate"].map(
    lambda value: f"{value:.2%}"
)
signal_check

,position_band,pages,median_impressions,median_days_since_update,observed_decline_rate
0,page_1,9215,2946.0,20.0,61.51%
1,striking,5876,1377.0,22.0,62.61%
2,page_3_5,6037,1209.0,22.0,58.41%
3,deep,878,427.0,22.0,31.78%


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] The task type, target/proxy, success metric, unit of analysis, and supported content action are explicit
- [x] Ready to commit under `work/notebooks/` and submit the repository URL